# LeetCode 2201 — Count Artifacts That Can Be Extracted bằng Python

Link đề: https://leetcode.com/problems/count-artifacts-that-can-be-extracted/ · Độ khó: **Medium** · Chủ đề: Array, Hash Table, Simulation

Skeleton LeetCode:

```python
class Solution:
    def digArtifacts(self, n: int, artifacts: List[List[int]], dig: List[List[int]]) -> int:
        ...
```

## 1. Mục tiêu học

1. Duyệt mọi ô của một hình chữ nhật `[r1..r2] × [c1..c2]` (**bao gồm** cả hai đầu) bằng hai vòng `range`.
2. Chọn cấu trúc tra cứu phù hợp: **danh sách** (`O(D)` mỗi lần hỏi) vs. **tập hợp băm** (`O(1)`) vs. **lưới boolean** (`O(n²)` bộ nhớ).
3. Mã hóa một ô `(r, c)` thành **một số nguyên** `r·n + c` (hoặc tuple) làm khóa, và hiểu vì sao mã hóa phải **đơn ánh**.
4. Cài cách "đảo chiều": ánh xạ **ô → vật** và **đếm ngược số ô còn chôn** của mỗi vật khi đào.
5. Kiểm thử đối chiếu nhiều cách trên dữ liệu ngẫu nhiên có ràng buộc "không chồng nhau, mỗi vật ≤ 4 ô" (seed cố định).

## 2. Đề bài & ví dụ

Có lưới `n × n` (đánh số từ 0) với một số **cổ vật** hình chữ nhật bị chôn. `artifacts[i] = [r1, c1, r2, c2]`: cổ vật thứ `i` chiếm mọi ô từ góc trên-trái `(r1, c1)` tới góc dưới-phải `(r2, c2)` (**gồm cả hai góc**).

Bạn đào các ô trong `dig` (`dig[i] = [r, c]`). Một cổ vật **lấy được** nếu **mọi** ô của nó đều đã được đào. Trả về số cổ vật lấy được.

Bảo đảm: các cổ vật **không chồng nhau**; mỗi cổ vật chiếm **tối đa 4 ô**; các ô trong `dig` **không trùng**.

**Ví dụ 1:** `n = 2, artifacts = [[0,0,0,0],[0,1,1,1]], dig = [[0,0],[0,1]]` → `1`. Cổ vật 0 (ô `(0,0)`) lấy được; cổ vật 1 (ô `(0,1)`, `(1,1)`) còn ô `(1,1)` chưa đào.

**Ví dụ 2:** cùng `n, artifacts`, `dig = [[0,0],[0,1],[1,1]]` → `2`.

**Ràng buộc:** `1 ≤ n ≤ 1000`, `1 ≤ len(artifacts), len(dig) ≤ min(n², 10⁵)`.

In [1]:
import random
import time
from typing import List

examples = [
    (2, [[0, 0, 0, 0], [0, 1, 1, 1]], [[0, 0], [0, 1]], 1),
    (2, [[0, 0, 0, 0], [0, 1, 1, 1]], [[0, 0], [0, 1], [1, 1]], 2),
]
def draw(n, artifacts, dig):
    g = [["." for _ in range(n)] for _ in range(n)]
    for idx, (r1, c1, r2, c2) in enumerate(artifacts):
        for r in range(r1, r2 + 1):
            for c in range(c1, c2 + 1):
                g[r][c] = str(idx)
    dug = {tuple(d) for d in dig}
    for r in range(n):
        print(" ".join(f"{g[r][c]}{'*' if (r, c) in dug else ' '}" for c in range(n)))

for n, arts, dig, exp in examples:
    draw(n, arts, dig); print("-> kỳ vọng", exp, "(số = mã cổ vật, * = đã đào)"); print()

0* 1*
.  1 
-> kỳ vọng 1 (số = mã cổ vật, * = đã đào)

0* 1*
.  1*
-> kỳ vọng 2 (số = mã cổ vật, * = đã đào)



## 3. Kiến thức nền

### 3.1 Duyệt các ô của hình chữ nhật (bao gồm hai đầu)

Ô `(r, c)` thuộc cổ vật ⇔ `r1 ≤ r ≤ r2` và `c1 ≤ c ≤ c2`. Vì `range(a, b)` **không** gồm `b`, phải dùng `range(r1, r2 + 1)`.

In [2]:
def cells(r1, c1, r2, c2):
    return [(r, c) for r in range(r1, r2 + 1) for c in range(c1, c2 + 1)]

print(cells(0, 1, 1, 1), "| một ô:", cells(3, 3, 3, 3), "| 2x2:", cells(0, 0, 1, 1))

[(0, 1), (1, 1)] | một ô: [(3, 3)] | 2x2: [(0, 0), (0, 1), (1, 0), (1, 1)]


### 3.2 `in` trên list vs. set

`x in list` duyệt tuần tự `O(len)`; `x in set` dùng băm, trung bình `O(1)`. Phần tử của `set` phải **hashable** — `list` thì không, nên đổi `[r, c]` thành `tuple` `(r, c)`.

In [3]:
L = [[i, i] for i in range(100000)]
S = {(i, i) for i in range(100000)}
t = time.perf_counter(); _ = [99999, 99999] in L; t1 = time.perf_counter() - t
t = time.perf_counter(); _ = (99999, 99999) in S; t2 = time.perf_counter() - t
print(f"list: {t1 * 1e6:.0f} µs | set: {t2 * 1e6:.1f} µs")

list: 1200 µs | set: 39.6 µs


### 3.3 Mã hóa ô thành một số nguyên `r·n + c`

Với `0 ≤ c < n`, ánh xạ `(r, c) ↦ r·n + c` là **đơn ánh** (giống viết số trong hệ cơ số `n`). Số nguyên băm nhanh hơn tuple một chút. Mã hóa như `r + c` hay `r·c` **không** đơn ánh — sẽ gây va chạm.

In [4]:
n = 4
codes = {(r, c): r * n + c for r in range(n) for c in range(n)}
print("đơn ánh?", len(set(codes.values())) == n * n, "| (2,3) ->", codes[(2, 3)], "-> giải mã:", divmod(11, n))
print("r + c đơn ánh?", len({r + c for r in range(n) for c in range(n)}) == n * n)

đơn ánh? True | (2,3) -> 11 -> giải mã: (2, 3)
r + c đơn ánh? False


### 3.4 `all` / `any`

`all(điều kiện cho mọi ô)` ⇒ "mọi ô đã đào". `any` ⇒ "có ít nhất một ô". Cả hai dừng sớm (short-circuit).

In [5]:
dug = {(0, 0), (0, 1)}
print(all(c in dug for c in cells(0, 1, 1, 1)), any(c in dug for c in cells(0, 1, 1, 1)))

False True


## 4. Ý tưởng nền tảng / trực giác

Ví dụ 1: tập ô đã đào `{(0,0), (0,1)}`.

| Cổ vật | Ô | Ô nào chưa đào | Lấy được? |
|---|---|---|---|
| 0: [0,0,0,0] | (0,0) | — | ✓ |
| 1: [0,1,1,1] | (0,1), (1,1) | (1,1) | ✗ |

Đáp án `1`.

Để thấy cái giá của mỗi cách lưu "đã đào", hãy đếm **số câu hỏi** "ô này đã đào chưa?": mỗi cổ vật hỏi tối đa 4 lần ⇒ tối đa `4·10⁵` câu hỏi. Nếu mỗi câu hỏi tốn `D = 10⁵` bước (duyệt list) thì tổng `4·10¹⁰` — quá chậm; nếu mỗi câu tốn `O(1)` (set/lưới) thì chỉ `4·10⁵`. Cấu trúc dữ liệu quyết định tất cả, còn "thuật toán" thì rất đơn giản.

Câu hỏi cốt lõi: "ô `(r, c)` đã đào chưa?" được hỏi **nhiều lần** (tổng số ô của mọi cổ vật ≤ `4·10⁵`), nên phải trả lời **nhanh** ⇒ dùng tập băm.

Góc nhìn ngược: mỗi lần đào một ô, nếu ô đó thuộc một cổ vật thì cổ vật đó **bớt một** ô còn chôn; khi về 0 thì lấy được.

## 5. Cách 1 — Brute Force / baseline: tra cứu bằng `in` trên danh sách `dig`

**Ý tưởng:** với mỗi cổ vật, kiểm tra mọi ô của nó có nằm trong **danh sách** `dig` không.

In [6]:
def dig_brute(n, artifacts, dig):
    ans = 0
    for r1, c1, r2, c2 in artifacts:
        if all([r, c] in dig for r in range(r1, r2 + 1) for c in range(c1, c2 + 1)):
            ans += 1
    return ans

for n, arts, dig, exp in examples:
    got = dig_brute(n, arts, dig); print(got, "kỳ vọng", exp); assert got == exp

1 kỳ vọng 1
2 kỳ vọng 2


**Độ phức tạp:** mỗi lần `in` tốn `O(D)` (`D = len(dig)`); tổng số lần hỏi ≤ `4A` ⇒ `O(A · D)` worst ≈ `4·10⁵ · 10⁵ = 4·10¹⁰` — TLE. Space `O(1)`.

**Nút thắt:** tra cứu tuyến tính trên danh sách.

In [7]:
rng = random.Random(0)
for D in (1000, 2000, 4000):
    n = 200
    dig = [[r, c] for r in range(n) for c in range(n)]; rng.shuffle(dig); dig = dig[:D]
    arts = [[r, c, r, c] for r, c in dig[-(D // 2):]]     # ô của cổ vật nằm cuối danh sách: `in` phải duyệt gần hết
    t = time.perf_counter(); dig_brute(n, arts, dig)
    print(f"A={len(arts)}, D={D}: {time.perf_counter() - t:.3f}s (gấp đôi cả hai -> ~x4)")

A=500, D=1000: 0.004s (gấp đôi cả hai -> ~x4)
A=1000, D=2000: 0.015s (gấp đôi cả hai -> ~x4)
A=2000, D=4000: 0.057s (gấp đôi cả hai -> ~x4)


## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1 (tra cứu nhanh):** chuyển `dig` thành **tập** `{(r, c)}` (hoặc `{r·n + c}`) ⇒ mỗi lần hỏi `O(1)` ⇒ tổng `O(D + 4A)`.

**Insight 2 (đảo chiều, đếm ngược):** ghi `owner[ô] = chỉ số cổ vật` và `remain[i] = số ô của cổ vật i`. Với mỗi ô đào: **nếu** ô có chủ **thì** `remain[owner] −= 1`; **nếu** về 0 **thì** `ans += 1`. Vì `dig` không trùng nên mỗi ô chỉ trừ một lần.

**Invariant (Cách đếm ngược):** sau khi xử lý `t` ô đào đầu tiên, `remain[i]` = số ô của cổ vật `i` **chưa** nằm trong `t` ô đó, và `ans` = số cổ vật có `remain = 0`.

**Quy tắc "nếu… thì…" cho Cách 2:** **nếu** tồn tại ô của cổ vật chưa đào **thì** cổ vật không lấy được (`all` dừng sớm).

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Tập các ô đã đào (bản nộp chính)

**Giải thích từng đoạn:**
1. `dug = {(r, c) for r, c in dig}`.
2. Với mỗi cổ vật, `all((r, c) in dug ...)` trên các ô của nó.
3. Đếm số cổ vật thỏa.

In [8]:
def dig_set(n, artifacts, dig):
    dug = {(r, c) for r, c in dig}
    return sum(all((r, c) in dug for r in range(r1, r2 + 1) for c in range(c1, c2 + 1))
               for r1, c1, r2, c2 in artifacts)

for n, arts, dig, exp in examples:
    assert dig_set(n, arts, dig) == exp
print([dig_set(*e[:3]) for e in examples])

[1, 2]


**Độ phức tạp Cách 2:** `O(D + 4A)` thời gian trung bình, `O(D)` bộ nhớ.

### 7.2 Cách 3 — Lưới boolean `n × n`

Thay tập băm bằng mảng 2 chiều `grid[r][c] = True` nếu đã đào. Tra cứu `O(1)` tuyệt đối (không phụ thuộc băm) nhưng tốn `n² = 10⁶` ô bộ nhớ khi `n = 1000` — vẫn chấp nhận được.

In [9]:
def dig_grid(n, artifacts, dig):
    grid = [[False] * n for _ in range(n)]
    for r, c in dig:
        grid[r][c] = True
    ans = 0
    for r1, c1, r2, c2 in artifacts:
        ok = True
        for r in range(r1, r2 + 1):
            row = grid[r]
            for c in range(c1, c2 + 1):
                if not row[c]:
                    ok = False; break
            if not ok: break
        ans += ok
    return ans

for n, arts, dig, exp in examples:
    assert dig_grid(n, arts, dig) == exp
print([dig_grid(*e[:3]) for e in examples])

[1, 2]


**Độ phức tạp Cách 3:** `O(n² + D + 4A)` thời gian (khởi tạo lưới), `O(n²)` bộ nhớ.

### 7.3 Cách 4 — Mã hóa số nguyên `r·n + c` trong tập

In [10]:
def dig_encoded(n, artifacts, dig):
    dug = {r * n + c for r, c in dig}
    ans = 0
    for r1, c1, r2, c2 in artifacts:
        if all(r * n + c in dug for r in range(r1, r2 + 1) for c in range(c1, c2 + 1)):
            ans += 1
    return ans

for n, arts, dig, exp in examples:
    assert dig_encoded(n, arts, dig) == exp
print([dig_encoded(*e[:3]) for e in examples])

[1, 2]


**Độ phức tạp Cách 4:** như Cách 2, hằng số nhỏ hơn một chút (băm số nguyên nhanh hơn băm tuple).

### 7.4 Cách 5 — Ánh xạ ô → cổ vật, đếm ngược khi đào

**Giải thích từng đoạn:**
1. `owner = {}`, `remain = [số ô của mỗi cổ vật]`.
2. Với mỗi ô của cổ vật `i`: `owner[(r, c)] = i`.
3. Duyệt `dig`: nếu ô có chủ `i` thì `remain[i] −= 1`; nếu `remain[i] == 0` thì `ans += 1`.

Ưu điểm: biết được **thời điểm** (lần đào thứ mấy) mỗi cổ vật lấy được — hữu ích cho biến thể (bài tập 5).

In [11]:
def dig_countdown(n, artifacts, dig):
    owner = {}
    remain = []
    for i, (r1, c1, r2, c2) in enumerate(artifacts):
        remain.append((r2 - r1 + 1) * (c2 - c1 + 1))
        for r in range(r1, r2 + 1):
            for c in range(c1, c2 + 1):
                owner[(r, c)] = i
    ans = 0
    for r, c in dig:
        i = owner.get((r, c))
        if i is not None:
            remain[i] -= 1
            if remain[i] == 0:
                ans += 1
    return ans

for n, arts, dig, exp in examples:
    assert dig_countdown(n, arts, dig) == exp
print([dig_countdown(*e[:3]) for e in examples])

[1, 2]


**Độ phức tạp Cách 5:** `O(4A + D)` thời gian, `O(4A)` bộ nhớ (map ô → chủ). Lưu ý cần giả thiết "cổ vật không chồng nhau" — nếu chồng nhau, một ô có nhiều chủ và `owner` dạng `dict` đơn sẽ sai (xem biến thể ở mục 15).

## 8. TRACE từng bước

Trace Cách 5 trên ví dụ 2: `artifacts = [[0,0,0,0],[0,1,1,1]]`, `dig = [[0,0],[0,1],[1,1]]`.

Khởi tạo: `owner = {(0,0):0, (0,1):1, (1,1):1}`, `remain = [1, 2]`.

| Lần đào | Ô | Chủ | remain sau | Vừa lấy được? | ans |
|---|---|---|---|---|---|
| 1 | (0,0) | 0 | [0, 2] | cổ vật 0 | 1 |
| 2 | (0,1) | 1 | [0, 1] | — | 1 |
| 3 | (1,1) | 1 | [0, 0] | cổ vật 1 | 2 |

In [12]:
def trace_countdown(n, artifacts, dig):
    owner, remain = {}, []
    for i, (r1, c1, r2, c2) in enumerate(artifacts):
        remain.append((r2 - r1 + 1) * (c2 - c1 + 1))
        for r in range(r1, r2 + 1):
            for c in range(c1, c2 + 1):
                owner[(r, c)] = i
    print("owner =", owner, "| remain =", remain)
    ans = 0
    for t, (r, c) in enumerate(dig, 1):
        i = owner.get((r, c))
        if i is None:
            print(f"lần {t}: đào ({r},{c}) -> đất trống"); continue
        remain[i] -= 1; done = remain[i] == 0; ans += done
        print(f"lần {t}: đào ({r},{c}) -> cổ vật {i}, remain={remain}", "-> LẤY ĐƯỢC" if done else "", f"| ans={ans}")
    return ans

print("Đáp án:", trace_countdown(*examples[1][:3]))

owner = {(0, 0): 0, (0, 1): 1, (1, 1): 1} | remain = [1, 2]
lần 1: đào (0,0) -> cổ vật 0, remain=[0, 2] -> LẤY ĐƯỢC | ans=1
lần 2: đào (0,1) -> cổ vật 1, remain=[0, 1]  | ans=1
lần 3: đào (1,1) -> cổ vật 1, remain=[0, 0] -> LẤY ĐƯỢC | ans=2
Đáp án: 2


In [13]:
def trace_set(n, artifacts, dig):
    dug = {(r, c) for r, c in dig}; ans = 0
    for i, (r1, c1, r2, c2) in enumerate(artifacts):
        cs = cells(r1, c1, r2, c2); missing = [x for x in cs if x not in dug]
        ok = not missing; ans += ok
        print(f"cổ vật {i} {cs}: thiếu {missing} -> {'lấy được' if ok else 'chưa'} | ans={ans}")
    return ans

n_, arts_, dig_ = 4, [[0, 0, 1, 1], [2, 0, 2, 3], [3, 3, 3, 3], [0, 3, 1, 3]], [[0, 0], [0, 1], [1, 0], [1, 1], [3, 3], [0, 3], [2, 1]]
draw(n_, arts_, dig_)
print("Đáp án:", trace_set(n_, arts_, dig_))

0* 0* .  3*
0* 0* .  3 
1  1* 1  1 
.  .  .  2*
cổ vật 0 [(0, 0), (0, 1), (1, 0), (1, 1)]: thiếu [] -> lấy được | ans=1
cổ vật 1 [(2, 0), (2, 1), (2, 2), (2, 3)]: thiếu [(2, 0), (2, 2), (2, 3)] -> chưa | ans=1
cổ vật 2 [(3, 3)]: thiếu [] -> lấy được | ans=2
cổ vật 3 [(0, 3), (1, 3)]: thiếu [(1, 3)] -> chưa | ans=2
Đáp án: 2


## 9. Code nộp LeetCode

In [14]:
class Solution:
    def digArtifacts(self, n: int, artifacts: List[List[int]], dig: List[List[int]]) -> int:
        dug = {r * n + c for r, c in dig}        # mã hóa ô (r, c) -> r*n + c (đơn ánh)
        ans = 0
        for r1, c1, r2, c2 in artifacts:
            if all(r * n + c in dug
                   for r in range(r1, r2 + 1)    # gồm cả r2
                   for c in range(c1, c2 + 1)):  # gồm cả c2
                ans += 1
        return ans

for n, arts, dig, exp in examples:
    assert Solution().digArtifacts(n, arts, dig) == exp
print("Ví dụ đề: OK")

Ví dụ đề: OK


## 10. Vì sao thuật toán đúng

**Cách 2/4 (tập).** Cổ vật `i` lấy được ⇔ mọi ô `(r, c)` với `r1 ≤ r ≤ r2`, `c1 ≤ c ≤ c2` đều được đào. Hai vòng `range(r1, r2+1)`, `range(c1, c2+1)` liệt kê **đúng** các ô đó. Tập `dug` chứa **đúng** các ô đã đào (mã hóa `r·n + c` đơn ánh vì `0 ≤ c < n`), nên `x in dug` trả lời chính xác. Vậy biểu thức `all` đúng bằng điều kiện "lấy được".

**Cách 5 (đếm ngược).** Vì các cổ vật không chồng nhau, mỗi ô có **tối đa một** chủ. Vì `dig` không trùng, mỗi ô của cổ vật `i` làm `remain[i]` giảm **đúng một lần** khi được đào. Do đó `remain[i] = 0` ⇔ mọi ô của nó đã được đào, và mỗi cổ vật được cộng vào `ans` đúng một lần (đúng lúc `remain` chạm 0).

## 11. Phân tích Time / Space

Gọi `A = len(artifacts)`, `D = len(dig)`, mỗi cổ vật ≤ 4 ô.

| Cách | Time | Space |
|---|---|---|
| 1. `in` trên list | `O(A · D)` | `O(1)` |
| 2. Tập tuple | `O(D + 4A)` | `O(D)` |
| 3. Lưới boolean | `O(n² + D + 4A)` | `O(n²)` |
| 4. Tập số nguyên | `O(D + 4A)` | `O(D)` |
| 5. Ô → chủ, đếm ngược | `O(4A + D)` | `O(4A)` |

In [15]:
def make_case(n, A, D, rng):
    """Sinh cổ vật không chồng nhau (mỗi cái <= 4 ô) và danh sách đào không trùng."""
    used = set(); arts = []
    shapes = [(1, 1), (1, 2), (2, 1), (2, 2), (1, 3), (3, 1), (1, 4), (4, 1)]
    tries = 0
    while len(arts) < A and tries < 20 * A:
        tries += 1
        h, w = rng.choice(shapes)
        if h > n or w > n: continue
        r1, c1 = rng.randint(0, n - h), rng.randint(0, n - w)
        cs = [(r, c) for r in range(r1, r1 + h) for c in range(c1, c1 + w)]
        if any(x in used for x in cs): continue
        used.update(cs); arts.append([r1, c1, r1 + h - 1, c1 + w - 1])
    allc = [[r, c] for r in range(n) for c in range(n)]
    dig = rng.sample(allc, min(D, n * n))
    return arts, dig

rng = random.Random(2201)
arts, dig = make_case(1000, 60000, 100000, rng)
print("A =", len(arts), "D =", len(dig))
for name, fn in [("set", dig_set), ("grid", dig_grid), ("encoded", dig_encoded), ("countdown", dig_countdown),
                 ("Solution", lambda n, a, d: Solution().digArtifacts(n, a, d))]:
    t = time.perf_counter(); r = fn(1000, arts, dig)
    print(f"{name:9s}: {r} ({time.perf_counter() - t:.3f}s)")

A = 60000 D = 100000
set      : 1013 (0.067s)
grid     : 1013 (0.032s)
encoded  : 1013 (0.045s)


countdown: 1013 (0.079s)
Solution : 1013 (0.044s)


**So sánh bộ nhớ thực tế.** Độ phức tạp `O(D)` và `O(n²)` nghe giống nhau khi `D ≈ n²`, nhưng hằng số khác xa: một `set` các tuple tốn hàng chục byte mỗi phần tử (tuple + băm), còn lưới boolean dạng list-of-list tốn ~8 byte mỗi ô (con trỏ tới `True/False` dùng chung). Ô dưới đo kích thước gần đúng với `D = 10⁵` ô đào trên lưới `n = 1000`.

In [16]:
import sys as _sys
rng = random.Random(11)
n_mem = 1000
dig_mem = rng.sample([(r, c) for r in range(n_mem) for c in range(n_mem)], 100000)
s_tuple = set(dig_mem)
s_int = {r * n_mem + c for r, c in dig_mem}
grid_mem = [[False] * n_mem for _ in range(n_mem)]
size_tuple = _sys.getsizeof(s_tuple) + sum(_sys.getsizeof(t) for t in dig_mem)
size_int = _sys.getsizeof(s_int) + sum(_sys.getsizeof(x) for x in s_int)
size_grid = _sys.getsizeof(grid_mem) + sum(_sys.getsizeof(row) for row in grid_mem)
print(f"set tuple ~{size_tuple / 1e6:.1f} MB | set int ~{size_int / 1e6:.1f} MB | lưới {n_mem}x{n_mem} ~{size_grid / 1e6:.1f} MB")

set tuple ~9.8 MB | set int ~7.0 MB | lưới 1000x1000 ~8.1 MB


## 12. Bộ test

- Ví dụ đề.
- Biên: `n = 1`; cổ vật 1×4 / 4×1 / 2×2; đào hết lưới (mọi cổ vật lấy được); không đào trúng ô nào; cổ vật ở mép lưới (hàng/cột `n − 1`).
- Đối chiếu mọi cách với brute force trên dữ liệu ngẫu nhiên đúng ràng buộc (không chồng, ≤ 4 ô, `dig` không trùng; seed cố định).

In [17]:
fns = {
    "brute": dig_brute, "set": dig_set, "grid": dig_grid, "encoded": dig_encoded, "countdown": dig_countdown,
    "Solution": lambda n, a, d: Solution().digArtifacts(n, a, d),
}
edge = [
    (1, [[0, 0, 0, 0]], [[0, 0]], 1),
    (1, [[0, 0, 0, 0]], [], 0),
    (4, [[0, 0, 0, 3], [1, 0, 3, 0], [2, 2, 3, 3]], [[r, c] for r in range(4) for c in range(4)], 3),
    (4, [[0, 0, 0, 3]], [[0, 0], [0, 1], [0, 2]], 0),
    (3, [[2, 2, 2, 2], [0, 2, 1, 2]], [[2, 2], [1, 2]], 1),
]
for n, arts, dig, exp in [(n, a, d, e) for n, a, d, e in examples] + edge:
    for name, fn in fns.items():
        assert fn(n, arts, dig) == exp, (name, n, arts, dig)
print("Ví dụ + biên: OK")

Ví dụ + biên: OK


In [18]:
rng = random.Random(42)
for _ in range(400):
    n = rng.randint(1, 6)
    arts, dig = make_case(n, rng.randint(1, 6), rng.randint(0, n * n), rng)
    if not arts: continue
    want = dig_brute(n, arts, dig)
    for name, fn in fns.items():
        assert fn(n, arts, dig) == want, (name, n, arts, dig)
print("400 test ngẫu nhiên: mọi cách khớp brute force")

400 test ngẫu nhiên: mọi cách khớp brute force


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Dùng `range(r1, r2)` (bỏ mất hàng/cột cuối)

Với cổ vật một ô `[0,0,0,0]`, `range(0, 0)` rỗng ⇒ `all([])` là `True` ⇒ cổ vật bị coi là "lấy được" dù chưa đào.

In [19]:
def wrong_exclusive_range(n, artifacts, dig):
    dug = {(r, c) for r, c in dig}
    return sum(all((r, c) in dug for r in range(r1, r2) for c in range(c1, c2))    # SAI: thiếu +1
               for r1, c1, r2, c2 in artifacts)

n, arts, dig = 2, [[0, 0, 0, 0], [0, 1, 1, 1]], []
print("sai:", wrong_exclusive_range(n, arts, dig), "| đúng:", dig_set(n, arts, dig))

sai: 2 | đúng: 0


In [20]:
print("sửa (range(r1, r2 + 1)):", dig_set(n, arts, dig))

sửa (range(r1, r2 + 1)): 0


### Lỗi 2 — Dùng `any` thay vì `all` (đào trúng **một** ô là tính)

In [21]:
def wrong_any(n, artifacts, dig):
    dug = {(r, c) for r, c in dig}
    return sum(any((r, c) in dug for r in range(r1, r2 + 1) for c in range(c1, c2 + 1))   # SAI
               for r1, c1, r2, c2 in artifacts)

n, arts, dig, exp = examples[0]
print("sai:", wrong_any(n, arts, dig), "| đúng:", exp)

sai: 2 | đúng: 1


In [22]:
print("sửa (all):", dig_set(n, arts, dig))

sửa (all): 1


### Lỗi 3 — Mã hóa ô không đơn ánh (`r + c`)

`(0, 1)` và `(1, 0)` cùng mã `1` ⇒ đào `(1, 0)` bị hiểu là đã đào `(0, 1)`.

In [23]:
def wrong_encoding(n, artifacts, dig):
    dug = {r + c for r, c in dig}                                               # SAI: va chạm
    return sum(all(r + c in dug for r in range(r1, r2 + 1) for c in range(c1, c2 + 1))
               for r1, c1, r2, c2 in artifacts)

n, arts, dig = 2, [[0, 1, 0, 1]], [[1, 0]]
print("sai:", wrong_encoding(n, arts, dig), "| đúng:", dig_set(n, arts, dig))

sai: 1 | đúng: 0


In [24]:
print("sửa (r*n + c):", dig_encoded(n, arts, dig))

sửa (r*n + c): 0


### Lỗi 4 — Đếm ngược nhưng cộng `ans` mỗi khi `remain ≤ 0` (đếm nhiều lần) hoặc khởi tạo `remain` sai

Nếu tính số ô là `(r2 − r1)·(c2 − c1)` (quên `+1`), cổ vật một hàng có `remain = 0` ngay từ đầu và không bao giờ được cộng — hoặc với điều kiện `<= 0` thì bị cộng nhiều lần.

In [25]:
def wrong_remain(n, artifacts, dig):
    owner, remain = {}, []
    for i, (r1, c1, r2, c2) in enumerate(artifacts):
        remain.append((r2 - r1) * (c2 - c1))                                   # SAI: thiếu +1
        for r in range(r1, r2 + 1):
            for c in range(c1, c2 + 1):
                owner[(r, c)] = i
    ans = 0
    for r, c in dig:
        i = owner.get((r, c))
        if i is not None:
            remain[i] -= 1
            if remain[i] <= 0: ans += 1                                        # SAI: có thể cộng nhiều lần
    return ans

n, arts, dig, exp = examples[1]
print("sai:", wrong_remain(n, arts, dig), "| đúng:", exp)

sai: 3 | đúng: 2


In [26]:
print("sửa:", dig_countdown(n, arts, dig))

sửa: 2


### Lỗi 5 — Bỏ thẳng `[r, c]` (list) vào `set`

`list` **không hashable** nên `{[0, 0]}` gây `TypeError`. Ta bắt lỗi để xem thông báo (không để notebook dừng), rồi sửa bằng cách đổi sang `tuple` hoặc mã hóa số nguyên.

In [27]:
def wrong_list_in_set(dig):
    try:
        return {d for d in dig}                                  # SAI: phần tử là list
    except TypeError as e:
        return f"TypeError: {e}"

print("sai:", wrong_list_in_set([[0, 0], [0, 1]]))

sai: TypeError: unhashable type: 'list'


In [28]:
print("sửa:", {tuple(d) for d in [[0, 0], [0, 1]]}, "hoặc", {r * 2 + c for r, c in [[0, 0], [0, 1]]})

sửa: {(0, 1), (0, 0)} hoặc {0, 1}


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **"Tập các ô đã chọn" + nhiều câu hỏi "ô này có trong tập không"** ⇒ `set` (tuple hoặc số nguyên mã hóa).
- **Hình chữ nhật trên lưới** ⇒ nhớ `+1` ở đầu mút (đề cho tọa độ **bao gồm**).
- Mã hóa ô: `r·n + c` (hoặc `r·(maxC+1) + c`) — kiểm tra tính đơn ánh.
- Khi cần biết **lúc nào** một đối tượng hoàn thành ⇒ **đảo chiều**: ô → đối tượng, đếm ngược.
- Ràng buộc nhỏ "mỗi vật ≤ 4 ô" ⇒ duyệt ô của từng vật là rẻ.

## 15. Biến thể của bài

1. **Thời điểm lấy được mỗi cổ vật** (bài tập 5): Cách 5 ghi lại chỉ số lần đào.
2. **Cổ vật có thể chồng nhau:** `owner` thành `dict` ô → **danh sách** chủ; mỗi lần đào giảm `remain` của **mọi** chủ.
3. **Cổ vật lớn (không giới hạn 4 ô):** đếm số ô đã đào trong hình chữ nhật bằng **tổng tiền tố 2 chiều** `O(1)` mỗi cổ vật thay cho duyệt từng ô.
4. **Truy vấn trực tuyến:** sau mỗi lần đào, trả về số cổ vật đã lấy được — Cách 5 cho ngay.

Minh họa biến thể 3 (tự dựng): tiền tố 2 chiều trên lưới đào, kiểm bằng Cách 2 với cổ vật lớn.

In [29]:
def dig_prefix2d(n, artifacts, dig):
    P = [[0] * (n + 1) for _ in range(n + 1)]
    g = [[0] * n for _ in range(n)]
    for r, c in dig: g[r][c] = 1
    for r in range(n):
        for c in range(n):
            P[r + 1][c + 1] = g[r][c] + P[r][c + 1] + P[r + 1][c] - P[r][c]
    ans = 0
    for r1, c1, r2, c2 in artifacts:
        area = (r2 - r1 + 1) * (c2 - c1 + 1)
        got = P[r2 + 1][c2 + 1] - P[r1][c2 + 1] - P[r2 + 1][c1] + P[r1][c1]
        ans += got == area
    return ans

rng = random.Random(7)
for _ in range(300):
    n = rng.randint(1, 8)
    arts = []
    for _ in range(rng.randint(1, 5)):
        r1, c1 = rng.randint(0, n - 1), rng.randint(0, n - 1)
        arts.append([r1, c1, rng.randint(r1, n - 1), rng.randint(c1, n - 1)])   # có thể lớn và chồng nhau
    dig = rng.sample([[r, c] for r in range(n) for c in range(n)], rng.randint(0, n * n))
    assert dig_prefix2d(n, arts, dig) == dig_set(n, arts, dig)
print("Tiền tố 2 chiều (cổ vật lớn, cho phép chồng) khớp Cách 2")

Tiền tố 2 chiều (cổ vật lớn, cho phép chồng) khớp Cách 2


Minh họa biến thể 2 — **cổ vật được phép chồng nhau** (tự dựng). Một ô có thể thuộc nhiều cổ vật, nên `owner[ô]` là **danh sách** chủ; mỗi lần đào giảm `remain` của **mọi** chủ. Kiểm bằng Cách 2 (vốn không cần giả thiết "không chồng").

In [30]:
from collections import defaultdict

def dig_countdown_overlap(n, artifacts, dig):
    owners = defaultdict(list); remain = []
    for i, (r1, c1, r2, c2) in enumerate(artifacts):
        remain.append((r2 - r1 + 1) * (c2 - c1 + 1))
        for r in range(r1, r2 + 1):
            for c in range(c1, c2 + 1):
                owners[(r, c)].append(i)
    ans = 0
    for r, c in dig:
        for i in owners.get((r, c), ()):
            remain[i] -= 1
            ans += remain[i] == 0
    return ans

rng = random.Random(8)
for _ in range(300):
    n = rng.randint(1, 6); arts = []
    for _ in range(rng.randint(1, 6)):
        r1, c1 = rng.randint(0, n - 1), rng.randint(0, n - 1)
        arts.append([r1, c1, min(n - 1, r1 + rng.randint(0, 1)), min(n - 1, c1 + rng.randint(0, 1))])
    dig = rng.sample([[r, c] for r in range(n) for c in range(n)], rng.randint(0, n * n))
    assert dig_countdown_overlap(n, arts, dig) == dig_set(n, arts, dig)
print("Đếm ngược với cổ vật chồng nhau khớp Cách 2")
print("Còn Cách 5 gốc (owner một chủ) trên dữ liệu chồng nhau:",
      dig_countdown(2, [[0, 0, 0, 1], [0, 1, 0, 1]], [[0, 0], [0, 1]]), "vs đúng", dig_set(2, [[0, 0, 0, 1], [0, 1, 0, 1]], [[0, 0], [0, 1]]))

Đếm ngược với cổ vật chồng nhau khớp Cách 2
Còn Cách 5 gốc (owner một chủ) trên dữ liệu chồng nhau: 1 vs đúng 2


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. `in` trên list | brute force | `O(A·D)` | `O(1)` | dữ liệu rất nhỏ |
| 2. Tập tuple | tra cứu băm | `O(D + 4A)` | `O(D)` | dễ đọc |
| 3. Lưới boolean | mảng 2 chiều | `O(n² + D + A)` | `O(n²)` | `n` nhỏ, muốn tránh băm |
| 4. Tập số nguyên | mã hóa `r·n+c` | `O(D + 4A)` | `O(D)` | **nộp bài** |
| 5. Đếm ngược | ô → chủ | `O(4A + D)` | `O(4A)` | cần thời điểm / trực tuyến |

## 17. Bài tập (từ dễ đến khó)

1. **Đoán output:** `n = 3, artifacts = [[0,0,0,2],[1,1,2,1]], dig = [[0,0],[0,1],[0,2],[1,1]]`.
2. **Tính tay:** trace Cách 5 cho bài 1 (bảng `remain` sau mỗi lần đào).
3. **Điền TODO:** `artifact_cells(a)` trả về danh sách ô của cổ vật `a = [r1, c1, r2, c2]`.
4. **Tự viết lại:** viết lại Cách 4 không nhìn code.
5. **Mini challenge:** `extract_times(n, artifacts, dig)` — trả về danh sách, phần tử `i` là **số thứ tự lần đào** (1-indexed) khiến cổ vật `i` lấy được, hoặc `-1`. Kiểm bằng brute force (thử từng tiền tố của `dig`).

In [31]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def artifact_cells(a):
    # TODO
    pass

In [32]:
# Bài 4 — TODO (chỉ định nghĩa, không gọi)
def my_dig_artifacts(n, artifacts, dig):
    # TODO
    pass

In [33]:
# Bài 5 — TODO (chỉ định nghĩa, không gọi)
def extract_times(n, artifacts, dig):
    # TODO: owner + remain như Cách 5, ghi t khi remain chạm 0
    pass

## 18. Gợi ý 3 mức

**Bài 1.** (1) Cổ vật 0 chiếm `(0,0),(0,1),(0,2)` — đào đủ. (2) Cổ vật 1 chiếm `(1,1),(2,1)` — thiếu `(2,1)`. (3) Đáp án `1`.

**Bài 2.** (1) `remain = [3, 2]`. (2) Sau 3 lần đầu: `[0, 2]`. (3) Lần 4: `[0, 1]`.

**Bài 3.** (1) Hai vòng `for`. (2) `range(r1, r2 + 1)`. (3) `range(c1, c2 + 1)`.

**Bài 4.** (1) `dug = {r*n + c ...}`. (2) `all(...)`. (3) Đếm.

**Bài 5.** (1) `times = [-1] * A`. (2) Duyệt `dig` với `enumerate(dig, 1)`. (3) Khi `remain[i]` về 0, `times[i] = t`.

## 19. Đáp án tham khảo

In [34]:
# Bài 1, 2
b1 = (3, [[0, 0, 0, 2], [1, 1, 2, 1]], [[0, 0], [0, 1], [0, 2], [1, 1]])
assert Solution().digArtifacts(*b1) == 1
print("Bài 1: 1")
trace_countdown(*b1)

Bài 1: 1
owner = {(0, 0): 0, (0, 1): 0, (0, 2): 0, (1, 1): 1, (2, 1): 1} | remain = [3, 2]
lần 1: đào (0,0) -> cổ vật 0, remain=[2, 2]  | ans=0
lần 2: đào (0,1) -> cổ vật 0, remain=[1, 2]  | ans=0
lần 3: đào (0,2) -> cổ vật 0, remain=[0, 2] -> LẤY ĐƯỢC | ans=1
lần 4: đào (1,1) -> cổ vật 1, remain=[0, 1]  | ans=1


1

In [35]:
# Bài 3, 4
def artifact_cells_ans(a):
    r1, c1, r2, c2 = a
    return [(r, c) for r in range(r1, r2 + 1) for c in range(c1, c2 + 1)]

my_dig_artifacts_ans = dig_encoded
assert artifact_cells_ans([0, 1, 1, 1]) == [(0, 1), (1, 1)] and len(artifact_cells_ans([0, 0, 1, 1])) == 4
rng = random.Random(3)
for _ in range(200):
    n = rng.randint(1, 6); arts, dig = make_case(n, rng.randint(1, 5), rng.randint(0, n * n), rng)
    if arts: assert my_dig_artifacts_ans(n, arts, dig) == dig_brute(n, arts, dig)
print("Bài 3, 4: OK")

Bài 3, 4: OK


In [36]:
# Bài 5
def extract_times_ans(n, artifacts, dig):
    owner, remain = {}, []
    for i, a in enumerate(artifacts):
        cs = artifact_cells_ans(a); remain.append(len(cs))
        for x in cs: owner[x] = i
    times = [-1] * len(artifacts)
    for t, (r, c) in enumerate(dig, 1):
        i = owner.get((r, c))
        if i is not None:
            remain[i] -= 1
            if remain[i] == 0: times[i] = t
    return times

def extract_times_brute(n, artifacts, dig):
    res = []
    for a in artifacts:
        cs = artifact_cells_ans(a)
        t_found = -1
        for t in range(1, len(dig) + 1):
            dug = {tuple(d) for d in dig[:t]}
            if all(x in dug for x in cs): t_found = t; break
        res.append(t_found)
    return res

rng = random.Random(5)
for _ in range(200):
    n = rng.randint(1, 5); arts, dig = make_case(n, rng.randint(1, 5), rng.randint(0, n * n), rng)
    if arts: assert extract_times_ans(n, arts, dig) == extract_times_brute(n, arts, dig)
print("Bài 5: OK; ví dụ 2 ->", extract_times_ans(*examples[1][:3]))

Bài 5: OK; ví dụ 2 -> [1, 3]


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template:**

```python
dug = {r * n + c for r, c in dig}
return sum(all(r * n + c in dug
               for r in range(r1, r2 + 1) for c in range(c1, c2 + 1))
           for r1, c1, r2, c2 in artifacts)
```

In [37]:
def template(n, artifacts, dig):
    dug = {r * n + c for r, c in dig}
    return sum(all(r * n + c in dug for r in range(r1, r2 + 1) for c in range(c1, c2 + 1))
               for r1, c1, r2, c2 in artifacts)
print([template(*e[:3]) for e in examples])

[1, 2]


**Checklist:**

- [ ] `range(r1, r2 + 1)` và `range(c1, c2 + 1)` (tọa độ **bao gồm**).
- [ ] `all` (mọi ô) chứ không `any`.
- [ ] Tra cứu bằng `set` (tuple hoặc `r·n + c`), không dùng `in` trên list.
- [ ] Mã hóa ô đơn ánh.
- [ ] Nếu đếm ngược: số ô = `(r2−r1+1)(c2−c1+1)`, cộng `ans` đúng một lần khi về 0.
- [ ] Đã thử: `n = 1`, không đào gì, đào hết lưới.